In [36]:
import os

APP_DIR = "/content/uv_guard"
os.makedirs(APP_DIR, exist_ok=True)

print("OK:", APP_DIR)

OK: /content/uv_guard


In [37]:
html = """
<!DOCTYPE html>
<html lang="ja">

<head>

<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">

<title>UVガードナビ</title>

<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>

<link rel="stylesheet" href="style.css">

</head>

<body>

<div class="container">

<h1>☀️ UVガードナビ</h1>

<div class="card">

<label>肌タイプ</label>
<select id="skinType">
<option value="1">タイプ1（とても色白）</option>
<option value="2">タイプ2（色白）</option>
<option value="3">タイプ3（普通）</option>
<option value="4">タイプ4（やや色黒）</option>
<option value="5">タイプ5（色黒）</option>
<option value="6">タイプ6（かなり色黒）</option>
</select>

<label>SPF</label>
<input type="number" id="spf" value="30">

<label>PA</label>
<select id="pa">
<option value="1">PA+</option>
<option value="1.2">PA++</option>
<option value="1.4">PA+++</option>
<option value="1.6">PA++++</option>
</select>

<button onclick="getLocation()">
📍 現在地から診断
</button>

</div>

<div class="result">

<h2>現在のUV情報</h2>

<p id="uv">-</p>

<p id="danger">-</p>

<p id="time">-</p>

<hr>

<h2>🟢 今日の安全時間帯</h2>

<p id="safeTime">-</p>

<hr>

<h2>⚠️ UVアラート</h2>

<p id="alert">-</p>

<hr>

<h2>📈 今日のUV変化</h2>

<canvas id="chart"></canvas>

</div>

</div>

<script src="script.js"></script>

</body>

</html>
"""

with open(f"{APP_DIR}/index.html","w",encoding="utf-8") as f:
    f.write(html)

print("index.html OK")

index.html OK


In [38]:
css = """
body{
    margin:0;
    padding:0;
    font-family:"Yu Gothic","Meiryo",sans-serif;
    background:#f4f6f9;
}

.container{
    max-width:700px;
    margin:30px auto;
    padding:20px;
}

h1{
    text-align:center;
    color:#ff9800;
    margin-bottom:20px;
}

.card,
.result{
    background:#ffffff;
    padding:20px;
    border-radius:15px;
    box-shadow:0 2px 8px rgba(0,0,0,0.15);
    margin-bottom:20px;
}

label{
    display:block;
    margin-top:15px;
    margin-bottom:5px;
    font-weight:bold;
}

select,
input{
    width:100%;
    padding:10px;
    font-size:16px;
    border:1px solid #cccccc;
    border-radius:8px;
    box-sizing:border-box;
}

button{
    width:100%;
    margin-top:20px;
    padding:12px;
    font-size:18px;
    background:#ff9800;
    color:white;
    border:none;
    border-radius:10px;
    cursor:pointer;
}

button:hover{
    background:#f57c00;
}

.result h2{
    color:#333333;
    margin-top:10px;
}

.result p{
    font-size:18px;
    line-height:1.8;
}

hr{
    margin:20px 0;
    border:none;
    border-top:1px solid #dddddd;
}

#safeTime{
    color:#2e7d32;
    font-weight:bold;
}

#alert{
    color:#d32f2f;
    font-weight:bold;
}

canvas{
    width:100% !important;
    max-height:350px;
}
"""

with open(f"{APP_DIR}/style.css","w",encoding="utf-8") as f:
    f.write(css)

print("style.css OK")

style.css OK


In [43]:
js = """
let chart;

async function getLocation(){

 if(!navigator.geolocation){
   alert("位置情報が利用できません");
   return;
 }

 navigator.geolocation.getCurrentPosition(async(pos)=>{

  const lat=pos.coords.latitude;
  const lon=pos.coords.longitude;

  // Open-Meteo APIからUV指数を取得
  const url=`https://api.open-meteo.com/v1/forecast?latitude=${lat}&longitude=${lon}&current=uv_index&hourly=uv_index`;

  const res=await fetch(url);
  const data=await res.json();

  // 現在のUV指数
  const uv=data.current.uv_index;

  const skin=Number(document.getElementById("skinType").value);
  const spf=Number(document.getElementById("spf").value);
  const pa=Number(document.getElementById("pa").value);

  const base={1:10,2:15,3:20,4:30,5:40,6:60};

  // 日焼け開始時間の計算
  const burn=(base[skin]/Math.max(uv,1))*spf*pa;

  let danger="低";
  if(uv>=11) danger="極端";
  else if(uv>=8) danger="非常に高い";
  else if(uv>=6) danger="高";
  else if(uv>=3) danger="中";

  document.getElementById("uv").innerHTML=
  `UV指数：<b>${uv.toFixed(1)}</b>`;

  document.getElementById("danger").innerHTML=
  `危険度：<b>${danger}</b>`;

  document.getElementById("time").innerHTML=
  `日焼け開始：約<b>${Math.round(burn)}分</b>`;

  // 判定用：今日の「月/日」をローカル基準で取得
  const now = new Date();
  const todayMonthDay = `${now.getMonth() + 1}/${now.getDate()}`;

  const labels=[];
  const values=[];
  const safeToday=[];
  let alertMsg="今日は急上昇なし";

  for(let i=0; i<data.hourly.time.length; i++){
    const d = new Date(data.hourly.time[i]);

    // X軸のラベルを「月/日 時時」の形式にする（例: 7/7 12時）
    const monthDay = `${d.getMonth() + 1}/${d.getDate()}`;
    const label = `${monthDay} ${d.getHours()}時`;

    labels.push(label);
    values.push(data.hourly.uv_index[i]);

    // 「今日」のデータのみ、安全時間帯とアラートの判定を行う
    if(monthDay === todayMonthDay) {
      if(data.hourly.uv_index[i] <= 3){
        safeToday.push(d.getHours());
      }
      if(data.hourly.uv_index[i] >= 8 && alertMsg === "今日は急上昇なし"){
        alertMsg = `${d.getHours()}時頃 紫外線が非常に強くなります`;
      }
    }
  }

  // 今日の安全時間帯の表示
  if(safeToday.length == 0){
    document.getElementById("safeTime").innerHTML = "今日は安全時間帯なし";
  } else {
    let txt = "";
    for(let i=0; i<safeToday.length; i++){
      txt += safeToday[i] + "時 ";
    }
    document.getElementById("safeTime").innerHTML = txt;
  }

  // UVアラートの表示
  document.getElementById("alert").innerHTML = alertMsg;

  // グラフの描画・更新
  if(chart){
    chart.destroy();
  }

  chart=new Chart(document.getElementById("chart"),{
    type:"line",
    data:{
      labels:labels,
      datasets:[{
        label:"UV指数の推移",
        data:values,
        borderColor:"orange",
        backgroundColor:"rgba(255,165,0,0.2)",
        fill:true,
        tension:0.4
      }]
    },
    options:{
      responsive:true,
      scales:{
        y:{
          beginAtZero:true,
          max:12
        }
      }
    }
  });

 },()=>{
   alert("位置情報を取得できませんでした");
 });

}
"""

In [44]:
%cd /content/uv_guard
!python -m http.server 8000

/content/uv_guard
Serving HTTP on 0.0.0.0 port 8000 (http://0.0.0.0:8000/) ...
127.0.0.1 - - [07/Jul/2026 06:59:26] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Jul/2026 06:59:27] "GET /script.js HTTP/1.1" 200 -
127.0.0.1 - - [07/Jul/2026 06:59:27] "GET /style.css HTTP/1.1" 200 -
127.0.0.1 - - [07/Jul/2026 06:59:41] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Jul/2026 06:59:41] "GET /style.css HTTP/1.1" 200 -
127.0.0.1 - - [07/Jul/2026 06:59:41] "GET /script.js HTTP/1.1" 200 -

Keyboard interrupt received, exiting.
Traceback (most recent call last):
  File "/usr/lib/python3.12/http/server.py", line 1284, in test
    httpd.serve_forever()
  File "/usr/lib/python3.12/socketserver.py", line 235, in serve_forever
    ready = selector.select(poll_interval)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/selectors.py", line 415, in select
    fd_event_list = self._selector.poll(timeout)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt

During handling of the

In [39]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8000)"))

https://8000-m-s-kkb-use1d1-s20c1rihpcc0-d.us-east1-1.prod.colab.dev
